# Demo 4 - Scheduled enrichment job (parameterised)

**Workshop:** F16 Advanced analytics (notebooks / ML) · **Pool:** Medium · **Time:** ~30 min

This notebook is designed to run **as a scheduled job**. It reads sign-ins over a lookback
window, scores users by failed-sign-in pressure, and writes a compact **enrichment table** -
satisfying the F16 Definition of Done: *"a notebook-based analytic runs on a schedule over
lake data and produces a usable detection or enrichment output, with cost monitored."*

**Parameterise it:** select the cell below, open the cell menu, and choose
**Mark Cell as Parameters**. When you create the job, expand **Default parameters ->
Refresh parameters** to load these, and override per run if needed.

## Parameters (mark this cell as *Parameters*)

Right-click this cell and choose **Mark Cell as Parameters** before scheduling the
notebook. That tells the job runner these values can be overridden per run, so the same
notebook can be scheduled several times with different windows or thresholds.

The parameters are lowercase here, unlike the other notebooks, because that is the
convention the job scheduler expects.

In [ ]:
# Parameters
lookback_days = 7
min_failed_attempts = 10
workspace_name = "your-workspace-name"

## Compute the enrichment

The whole analysis in one cell, because a scheduled job wants to be one linear pass rather
than an interactive exploration.

Read both sign-in tables, derive success or failure from the Entra error code, and score
each user by failed-sign-in pressure. `SuspicionScore` deliberately multiplies failures by
distinct IPs: 100 failures from one address is a broken service account, while 100 failures
from 40 addresses is a distributed attack, and a plain failure count cannot tell them
apart.

`HasSuccessAfterFailures` is the column to sort by in a real queue. A user who failed
repeatedly and then succeeded is the one worth a phone call.

Users with no `UserPrincipalName` are dropped. A null-keyed row in a table the SOC consumes
is not actionable.

In [ ]:
from sentinel_lake.providers import MicrosoftSentinelProvider
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

data_provider = MicrosoftSentinelProvider(spark)

STATUS_SCHEMA = StructType([StructField("errorCode", StringType(), True)])
SUCCESS_CODES = ["0", "50125", "50140", "70043", "70044"]

def prepare(table_name):
    df = data_provider.read_table(table_name, workspace_name)
    df = df.filter(F.col("TimeGenerated") >= F.expr(f"current_timestamp() - INTERVAL {int(lookback_days)} DAYS"))
    df = (df.withColumn("Status_json", F.from_json(F.col("Status"), STATUS_SCHEMA))
            .withColumn("ResultCode", F.col("Status_json.errorCode"))
            .withColumn("Outcome", F.when(F.col("ResultCode").isin(SUCCESS_CODES), "Success").otherwise("Failure")))
    return df.select("UserPrincipalName", "IPAddress", "Outcome")

signins = prepare("SigninLogs")
try:
    signins = signins.unionByName(prepare("AADNonInteractiveUserSignInLogs"))
except Exception as e:
    print("Non-interactive table unavailable:", e)

# A null-keyed row in an enrichment table the SOC consumes isn't actionable - drop it here
# rather than shipping it to the custom table.
signins = signins.filter(F.col("UserPrincipalName").isNotNull() &
                         (F.trim(F.col("UserPrincipalName")) != ""))

enrichment = (signins.groupBy("UserPrincipalName")
    .agg(F.sum(F.when(F.col("Outcome") == "Failure", 1).otherwise(0)).alias("FailedSignins"),
         F.sum(F.when(F.col("Outcome") == "Success", 1).otherwise(0)).alias("SuccessfulSignins"),
         F.countDistinct("IPAddress").alias("DistinctIPs"))
    .filter(F.col("FailedSignins") >= min_failed_attempts)
    .withColumn("SuspicionScore",
                F.col("FailedSignins") * (F.col("DistinctIPs") + F.lit(1)))
    .withColumn("HasSuccessAfterFailures", (F.col("SuccessfulSignins") > 0).cast("int"))
    .withColumn("LookbackDays", F.lit(int(lookback_days)))
    .withColumn("GeneratedAt", F.current_timestamp())
    .orderBy(F.desc("SuspicionScore")))

print("Enrichment rows:", enrichment.count())
enrichment.show(20, truncate=False)

## Write the enrichment table

Lake tier (`_SPRK`), overwritten each run so the table always reflects the latest window.
Keep the expensive compute in the lake tier and persist only this compact result.

In [ ]:
# No database argument, so this lands in the "System tables" database: the only lake-tier
# location that supports overwrite. Read it back with
#     data_provider.read_table("SigninAnomalyEnrichment_SPRK")
run_id = data_provider.save_as_table(
    enrichment,
    "SigninAnomalyEnrichment_SPRK",
    write_options={"mode": "overwrite"},
)
print("Wrote SigninAnomalyEnrichment_SPRK, run id:", run_id)

# --- Optional: promote flagged users to the analytics tier for KQL hunting (append-only) ---
# data_provider.save_as_table(
#     enrichment, "SigninAnomalyEnrichment_SPRK_CL", workspace_name,
#     write_options={"mode": "append"})

## Schedule this as a job

1. Toolbar **Create schedule Job** (or right-click the file in Explorer ->
   **Microsoft Sentinel -> Create schedule Job**) -> **Use existing notebook**.
2. **Job details:** name `f16-identity-enrichment-daily`; **Spark pool:** Medium.
3. **Schedule:** *Scheduled -> Daily* at a quiet time (e.g. 02:00), or *On demand* to prove it
   now. Times are in your timezone.
4. Expand **Default parameters -> Refresh parameters**, review, **Submit**.
5. **Jobs** panel -> select the job -> **Run now** -> watch **Run history**.
6. Verify output: read `SigninAnomalyEnrichment_SPRK` in a notebook, or (if promoted) hunt
   `SigninAnomalyEnrichment_SPRK_CL` in KQL advanced hunting.

**Governance:** job timeout 8 h · max 3 concurrent jobs · everything visible in Run history.
Parameter precedence: notebook defaults -> job-config values -> runtime overrides.

**This is the F16 Definition of Done, met with real data.**